# 注意力机制

验证 Scaled Dot-Product Attention 和 Multi-Head Attention 的计算逻辑，并可视化注意力权重与 Causal Mask。

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} · 设备: {device}")

## 直觉

Query = 检索请求，Key = 索引标签，Value = 真实内容。注意力就是：用 Query 和每个 Key 算相关性，按相关性加权聚合 Value。

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

> **你要记住**：`QKᵀ` 算相关性，`softmax` 归一化成权重，`@ V` 做加权聚合。

In [ ]:
# Step 1：最小可运行版
# 按相关性做加权聚合
# softmax(QK^T / √d_k) @ V
# 时间 O(n²d)，空间 O(n²)

def attention(q, k, v, mask=None):
    """
    Args:
        q, k, v: (batch, heads, seq, d_k)
        mask:    (batch, 1, 1, seq_k) 或 None
    Returns:
        context: (batch, heads, seq, d_k)
        weights: (batch, heads, seq, seq)
    """
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

# 快速验证
q = torch.randn(1, 1, 6, 32)
k = torch.randn(1, 1, 6, 32)
v = torch.randn(1, 1, 6, 32)
ctx, w = attention(q, k, v)
print(f"context shape: {ctx.shape}")
print(f"权重行和（应全为 1.0）: {w[0,0].sum(dim=-1).tolist()}")

In [ ]:
# Step 2 & 3：Multi-Head Attention 完整实现
# 将 d_model 切分为 num_heads 个子空间，各自独立计算注意力
# MultiHead(Q,K,V) = Concat(head_1,...,head_h) W_O
# 时间 O(n²d)，与单头相同；头数不改变总计算量

import torch
import torch.nn as nn
import math

def attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, f"d_model={d_model} 不能被 num_heads={num_heads} 整除"
        self.h = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query, key, value: (batch, seq, d_model)
            mask: (batch, 1, 1, seq_k) 或 None
        Returns:
            out:     (batch, seq, d_model)
            weights: (batch, heads, seq_q, seq_k)
        """
        bsz = query.size(0)
        q = self.w_q(query).view(bsz, -1, self.h, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(bsz, -1, self.h, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(bsz, -1, self.h, self.d_k).transpose(1, 2)
        ctx, weights = attention(q, k, v, mask)
        ctx = ctx.transpose(1, 2).contiguous().view(bsz, -1, self.h * self.d_k)
        return self.w_o(ctx), weights

torch.manual_seed(42)
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)
out, weights = mha(x, x, x)
print(f"输入 shape:       {x.shape}")
print(f"输出 shape:       {out.shape}  （应与输入相同）")
print(f"注意力权重 shape: {weights.shape}  （batch, heads, seq, seq）")
print(f"参数量:           {sum(p.numel() for p in mha.parameters()):,}")

## 验证

检查输出 shape、权重行和、以及 Causal Mask 是否生效。

In [ ]:
# 验证 Causal Mask：位置 i 只能关注位置 0..i
import torch
import math

def attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

SEQ = 5
causal_mask = torch.tril(torch.ones(1, 1, SEQ, SEQ))
q = torch.randn(1, 1, SEQ, 16)
k = torch.randn(1, 1, SEQ, 16)
v = torch.randn(1, 1, SEQ, 16)
_, w = attention(q, k, v, mask=causal_mask)

print("Causal Mask:")
print(causal_mask[0, 0].int())
print("\n应用后注意力权重（上三角应为 0）:")
print(w[0, 0].detach().round(decimals=3))

## 可视化

In [ ]:
# 可视化各注意力头的权重热力图
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

def attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.h = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
    def forward(self, query, key, value, mask=None):
        bsz = query.size(0)
        q = self.w_q(query).view(bsz, -1, self.h, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(bsz, -1, self.h, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(bsz, -1, self.h, self.d_k).transpose(1, 2)
        ctx, weights = attention(q, k, v, mask)
        ctx = ctx.transpose(1, 2).contiguous().view(bsz, -1, self.h * self.d_k)
        return self.w_o(ctx), weights

torch.manual_seed(42)
tokens = ["The", "cat", "sat", "on", "the", "mat"]
SEQ = len(tokens)
mha = MultiHeadAttention(d_model=64, num_heads=4)
x = torch.randn(1, SEQ, 64)
_, attn = mha(x, x, x)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("各注意力头权重热力图（随机初始化）", fontsize=13)
for h, ax in enumerate(axes):
    w = attn[0, h].detach().numpy()
    im = ax.imshow(w, vmin=0, vmax=w.max(), cmap="YlOrBr")
    ax.set_xticks(range(SEQ))
    ax.set_yticks(range(SEQ))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_title(f"Head {h+1}")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
print("提示：真实训练后，不同头会关注语法、语义、位置等不同模式。")